# APIs vs MCP vs A2A — Hands-on Tutorial

Companion notebook for the deck `apis_mcp_a2a_course_deck_boston_analytics_v3.pptx`.

**Scenario:** *Open-source dependency health triage.* Same shape as the renewal-triage example in the deck, but every fact comes live from the **public GitHub REST API**. The LLM (Google Gemini) reasons across all three paths.

**Everything is in this notebook** — the MCP server, the MCP client, and the agent. No external files. The server runs **in-process** via `mcp.shared.memory.create_connected_server_and_client_session`, so you see the full protocol surface (`initialize` → `tools/list` → `tools/call`) without subprocess plumbing.

| Path | Who decides? | Exchange | Use when... |
|------|--------------|----------|-------------|
| **1. API**  | Your app | Typed HTTP calls | Capability is known, deterministic |
| **2. MCP**  | LangGraph agent (Gemini) over a real MCP server | `tools/list` + `tools/call` | AI needs governed access to changing context |
| **3. A2A**  | Peer agents | Tasks, messages, artifacts | Independent agents must collaborate |

## 0. Setup

In [ ]:
%pip install -q google-genai requests mcp langchain-mcp-adapters langchain-google-genai langgraph

In [ ]:
import os, json, time, uuid
from datetime import datetime, timezone, timedelta
from getpass import getpass
from dataclasses import dataclass, field
import requests
from google import genai
from google.genai import types

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Paste GEMINI_API_KEY: ")

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL = "gemini-2.5-flash"

GH_HEADERS = {"Accept": "application/vnd.github+json",
              "X-GitHub-Api-Version": "2022-11-28"}
if os.environ.get("GITHUB_TOKEN"):
    GH_HEADERS["Authorization"] = f"Bearer {os.environ['GITHUB_TOKEN']}"

REPO = "pallets/flask"   # try 'pandas-dev/pandas' (active) or 'openai/gpt-2' (dormant)
print(client.models.generate_content(model=MODEL, contents="reply 'ready'").text)

## 1. Four backend functions — live GitHub REST

Plain Python functions that hit `api.github.com`. Used as-is across all three architectures.

In [ ]:
GH = "https://api.github.com"

def gh_get(path: str, params: dict | None = None):
    r = requests.get(f"{GH}{path}", headers=GH_HEADERS, params=params, timeout=20)
    r.raise_for_status()
    return r.json()

def get_repo(repo: str) -> dict:
    """GitHub repo metadata: stars, forks, language, archived flag, last push, license. `repo` is 'owner/name'."""
    d = gh_get(f"/repos/{repo}")
    return {k: d.get(k) for k in (
        "full_name","description","language","stargazers_count","forks_count",
        "open_issues_count","pushed_at","created_at","archived","license")}

def get_recent_commits(repo: str, days: int = 90) -> dict:
    """Commits in last N days. Returns commit count, unique author count, latest message."""
    since = (datetime.now(timezone.utc) - timedelta(days=days)).isoformat()
    commits = gh_get(f"/repos/{repo}/commits", params={"since": since, "per_page": 100})
    authors = {c["commit"]["author"]["email"] for c in commits if c.get("commit",{}).get("author")}
    last_msg = commits[0]["commit"]["message"].splitlines()[0] if commits else ""
    return {"window_days": days, "commit_count": len(commits),
            "unique_authors": len(authors), "latest_message": last_msg[:140]}

def get_issue_summary(repo: str) -> dict:
    """Open-issue counts and ages. Use to assess support burden / staleness."""
    issues = gh_get(f"/repos/{repo}/issues", params={"state":"open","per_page":100})
    issues = [i for i in issues if "pull_request" not in i]
    now = datetime.now(timezone.utc)
    ages = [(now - datetime.fromisoformat(i["created_at"].replace("Z","+00:00"))).days for i in issues]
    return {"open_issue_count": len(issues),
            "median_age_days": int(sorted(ages)[len(ages)//2]) if ages else 0,
            "oldest_age_days": max(ages) if ages else 0,
            "sample_titles": [i["title"] for i in issues[:5]]}

def get_release_info(repo: str) -> dict:
    """Most-recent releases: tag, days since latest, count seen."""
    rels = gh_get(f"/repos/{repo}/releases", params={"per_page":10})
    if not rels:
        return {"release_count_total_seen":0,"latest_tag":None,"days_since_latest":None}
    latest = rels[0]
    days = (datetime.now(timezone.utc) - datetime.fromisoformat(latest["published_at"].replace("Z","+00:00"))).days
    return {"release_count_total_seen": len(rels),"latest_tag": latest["tag_name"],
            "days_since_latest": days, "latest_name": latest.get("name")}

for fn in (get_repo, get_recent_commits, get_issue_summary, get_release_info):
    print(fn.__name__, "->", json.dumps(fn(REPO), default=str)[:180])

---
## Path 1 — APIs: deterministic fetch + LLM drafting

App owns orchestration. Hand-coded scoring rule. Gemini is used **only** at the final step to write the Slack note from structured findings.

In [ ]:
def health_triage_via_api(repo: str) -> dict:
    t0 = time.time()
    info     = get_repo(repo)
    commits  = get_recent_commits(repo, days=90)
    issues   = get_issue_summary(repo)
    releases = get_release_info(repo)
    score = 0
    if info["archived"]:                                  score += 60
    if commits["commit_count"] == 0:                      score += 40
    elif commits["commit_count"] < 5:                     score += 20
    if commits["unique_authors"] <= 1:                    score += 15
    if issues["oldest_age_days"] > 365:                   score += 15
    if releases["days_since_latest"] is None:             score += 20
    elif releases["days_since_latest"] > 365:             score += 25
    verdict = "CRITICAL" if score >= 60 else "AT_RISK" if score >= 30 else "HEALTHY"
    findings = {"info":info,"commits":commits,"issues":issues,"releases":releases,"score":score,"verdict":verdict}
    findings["draft_message"] = client.models.generate_content(model=MODEL, contents=(
        "Write a short professional Slack message (<=120 words) from a platform team "
        "reporting dependency health to internal users. Cite specific signals; recommend one action. "
        f"JSON facts:\n{json.dumps(findings, default=str)}")).text.strip()
    findings["latency_sec"] = round(time.time() - t0, 2)
    findings["recommended_action"] = ("Schedule maintainer outreach + plan migration" if verdict=="CRITICAL"
        else "Monitor and pin current version" if verdict=="AT_RISK" else "Continue standard usage")
    return findings

result_api = health_triage_via_api(REPO)
print(f"verdict: {result_api['verdict']}  score: {result_api['score']}  latency: {result_api['latency_sec']}s")
print(f"action : {result_api['recommended_action']}")
print("\n--- LLM-drafted message ---\n" + result_api["draft_message"])

---
## Path 2 — MCP: real server + real client, all in this notebook

**The whole protocol surface, one notebook.**

1. Define a `FastMCP` server inline. Register tools with `@mcp_server.tool()`. Schemas auto-derived from type hints + docstrings.
2. Connect a real `ClientSession` to the server **in-process** with `create_connected_server_and_client_session`. No subprocess. The MCP `initialize` handshake runs.
3. Bridge tools to LangChain via `langchain-mcp-adapters`. A **LangGraph ReAct agent** (Gemini) drives `tools/call`s.

Same `mcp_server` object would also run over stdio or Streamable HTTP — different `mcp_server.run(transport=...)`. Tools and schemas don't change.

In [ ]:
# -------- Server: defined right here in the notebook --------
from mcp.server.fastmcp import FastMCP

mcp_server = FastMCP("dependency-health")

@mcp_server.tool()
def mcp_get_repo(repo: str) -> dict:
    """GitHub repo metadata: stars, forks, language, archived flag, last push, license. `repo` is 'owner/name'."""
    return get_repo(repo)

@mcp_server.tool()
def mcp_get_recent_commits(repo: str, days: int = 90) -> dict:
    """Commits in the last N days. Returns commit count, unique authors, latest message."""
    return get_recent_commits(repo, days)

@mcp_server.tool()
def mcp_get_issue_summary(repo: str) -> dict:
    """Open-issue counts and ages. Assesses support burden."""
    return get_issue_summary(repo)

@mcp_server.tool()
def mcp_get_release_info(repo: str) -> dict:
    """Most-recent releases: tag, days since latest, count."""
    return get_release_info(repo)

print("MCP server defined with 4 tools")

In [ ]:
# -------- Client: in-process MCP session, real protocol --------
from mcp.shared.memory import create_connected_server_and_client_session

async with create_connected_server_and_client_session(mcp_server) as session:
    listing = await session.list_tools()        # wire-level tools/list
    print("tools/list ->")
    for t in listing.tools:
        print(f"  - {t.name}: {t.description[:80]}")
    # also demo a single tools/call via the raw protocol:
    raw = await session.call_tool("mcp_get_repo", {"repo": REPO})
    print("\nraw tools/call mcp_get_repo ->", json.loads(raw.content[0].text)["full_name"])

In [ ]:
# -------- Agent: LangGraph ReAct over the in-process MCP server --------
from langchain_mcp_adapters.tools import load_mcp_tools
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

llm = ChatGoogleGenerativeAI(model=MODEL, google_api_key=os.environ["GEMINI_API_KEY"])

MCP_SYSTEM = (
    "You are a dependency-health analyst. Use the tools to gather evidence about the repo, "
    "then return ONLY valid JSON with keys: verdict (HEALTHY|AT_RISK|CRITICAL), score (0-100), "
    "reasoning (short), citations (list of tool names), recommended_action, draft_message (<=120 words). "
    "Decide which tools to call; do not assume any."
)

async def health_triage_via_mcp(repo: str, verbose: bool = True) -> dict:
    t0 = time.time()
    async with create_connected_server_and_client_session(mcp_server) as session:
        tools = await load_mcp_tools(session)
        agent = create_react_agent(llm, tools)
        result = await agent.ainvoke({"messages": [
            {"role":"system","content":MCP_SYSTEM},
            {"role":"user",  "content":f"Assess repo: {repo}"},
        ]})
        tool_calls = [tc for m in result["messages"] for tc in (getattr(m,"tool_calls",[]) or [])]
        if verbose:
            for tc in tool_calls: print(f"[mcp tools/call] {tc['name']}({tc['args']})")
        text = result["messages"][-1].content
        try:    payload = json.loads(text)
        except: payload = text
        return {"verdict_json": payload,
                "tool_trace": [{"tool":tc["name"],"args":tc["args"]} for tc in tool_calls],
                "latency_sec": round(time.time()-t0,2)}

result_mcp = await health_triage_via_mcp(REPO)
print(f"\nlatency: {result_mcp['latency_sec']}s   tool calls: {len(result_mcp['tool_trace'])}")
print("\n--- MCP final answer ---\n")
print(json.dumps(result_mcp["verdict_json"], indent=2)
      if isinstance(result_mcp['verdict_json'], dict) else result_mcp['verdict_json'])

**What just happened (real MCP, in-process):**
- `FastMCP` registered tools with auto-derived JSON schemas.
- `create_connected_server_and_client_session` linked a real `ClientSession` via in-memory streams. The MCP `initialize` handshake ran.
- `session.list_tools()` and `session.call_tool(...)` are wire-level protocol calls — exactly what Claude Desktop / VS Code / Cursor would issue.
- `load_mcp_tools(session)` adapted MCP tools into LangChain `Tool` objects.
- LangGraph's `create_react_agent` runs the loop; each tool call goes through the protocol.

Want this server externally? `mcp_server.run(transport="stdio")` in a `__main__` block. Tools and schemas unchanged.

---
## Path 3 — A2A: peer agents via tasks and artifacts

Three opaque specialists, each with own LLM reasoning. Orchestrator sees only Agent Cards + artifacts.

In [ ]:
@dataclass
class AgentCard:
    name: str; description: str; skills: list[str]

@dataclass
class Task:
    id: str; goal: str; input: dict
    status: str = "submitted"
    messages: list[str] = field(default_factory=list)
    artifact: dict | None = None

def gemini_json(prompt: str) -> dict:
    r = client.models.generate_content(model=MODEL, contents=prompt,
        config=types.GenerateContentConfig(response_mime_type="application/json"))
    return json.loads(r.text)

class CommitsAgent:
    card = AgentCard("commits-agent","Assesses commit cadence + contributor diversity.",["activity_health"])
    def handle(self, task):
        task.status="working"
        meta=get_repo(task.input["repo"]); recent=get_recent_commits(task.input["repo"],days=90)
        task.artifact = gemini_json(
            "Software supply-chain analyst. Decide commit-activity health.\n"
            f"repo_meta: {json.dumps(meta,default=str)}\nrecent: {json.dumps(recent)}\n"
            "Reply JSON {verdict in [healthy,at_risk,critical], reason, key_signals}.")
        task.status="completed"; return task

class IssuesAgent:
    card = AgentCard("issues-agent","Reviews issue volume, age, themes.",["support_health"])
    def handle(self, task):
        task.status="working"
        s=get_issue_summary(task.input["repo"])
        task.artifact = gemini_json(
            "Maintainer-experience analyst. Decide issue-tracker health; skim sample titles for themes.\n"
            f"issues: {json.dumps(s)}\nReply JSON {{verdict, reason, themes}}.")
        task.status="completed"; return task

class ReleasesAgent:
    card = AgentCard("releases-agent","Evaluates release cadence.",["release_health"])
    def handle(self, task):
        task.status="working"
        r=get_release_info(task.input["repo"])
        task.artifact = gemini_json(
            "Release-engineering analyst. Decide release-cadence health.\n"
            f"releases: {json.dumps(r)}\nReply JSON {{verdict, reason}}.")
        task.status="completed"; return task

In [ ]:
REGISTRY = {a.card.name: a for a in [CommitsAgent(),IssuesAgent(),ReleasesAgent()]}

def a2a_send(name, goal, payload):
    t = Task(id=str(uuid.uuid4())[:8], goal=goal, input=payload)
    print(f"  [A2A] -> {name}  task={t.id}  goal='{goal}'")
    done = REGISTRY[name].handle(t)
    print(f"  [A2A] <- {name}  task={t.id}  status={done.status}")
    return done

def health_triage_via_a2a(repo: str) -> dict:
    t0=time.time()
    print("Discovered Agent Cards:")
    for a in REGISTRY.values(): print(f"  - {a.card.name}: {a.card.skills}")
    print("\nDelegating sub-tasks...")
    c = a2a_send("commits-agent","assess activity health",{"repo":repo})
    i = a2a_send("issues-agent", "assess support burden", {"repo":repo})
    r = a2a_send("releases-agent","assess release cadence",{"repo":repo})
    final = gemini_json(
        "You are the orchestrator. Three specialist artifacts:\n"
        f"- commits: {c.artifact}\n- issues: {i.artifact}\n- releases: {r.artifact}\n"
        "Reply JSON {verdict in [HEALTHY,AT_RISK,CRITICAL], score 0-100, recommended_action, draft_message<=120 words}.")
    return {"final_artifact":final,
            "specialist_artifacts":{"commits":c.artifact,"issues":i.artifact,"releases":r.artifact},
            "latency_sec":round(time.time()-t0,2)}

result_a2a = health_triage_via_a2a(REPO)
print(f"\nlatency: {result_a2a['latency_sec']}s")
print("\n--- Final orchestrator artifact ---\n")
print(json.dumps(result_a2a["final_artifact"], indent=2))

## 4. Side-by-side comparison (computed live)

In [ ]:
def _safe(d,*ks,default="-"):
    for k in ks:
        if isinstance(d,dict) and k in d: d=d[k]
        else: return default
    return d
mcp_v = result_mcp["verdict_json"] if isinstance(result_mcp["verdict_json"], dict) else {}
a2a_v = result_a2a["final_artifact"]

print("="*78); print(f"REPO: {REPO}"); print("="*78)
print(f"{'Path':<6} {'Verdict':<10} {'Score':<6} {'Latency':<10} Notes")
print("-"*78)
print(f"{'API':<6} {result_api['verdict']:<10} {result_api['score']:<6} {str(result_api['latency_sec'])+'s':<10} rule + LLM drafts")
print(f"{'MCP':<6} {_safe(mcp_v,'verdict'):<10} {str(_safe(mcp_v,'score')):<6} {str(result_mcp['latency_sec'])+'s':<10} {len(result_mcp['tool_trace'])} tool calls, LangGraph")
print(f"{'A2A':<6} {_safe(a2a_v,'verdict'):<10} {str(_safe(a2a_v,'score')):<6} {str(result_a2a['latency_sec'])+'s':<10} 3 specialists + orchestrator")
print("\n--- API draft ---\n" + result_api['draft_message'])
print("\n--- MCP draft ---\n" + str(_safe(mcp_v,'draft_message')))
print("\n--- A2A draft ---\n" + str(_safe(a2a_v,'draft_message')))

## 5. Try a contrasting (dormant) repo

In [ ]:
OTHER = "openai/gpt-2"
print("="*60); print(f"Contrasting repo: {OTHER}"); print("="*60)
_api = health_triage_via_api(OTHER)
_mcp = await health_triage_via_mcp(OTHER, verbose=False)
_a2a = health_triage_via_a2a(OTHER)
_mcpv = _mcp['verdict_json'] if isinstance(_mcp['verdict_json'],dict) else {}
_a2av = _a2a['final_artifact']
print(f"\n{'Path':<6} {'Verdict':<10} {'Score':<6} {'Latency':<10}")
print("-"*40)
print(f"{'API':<6} {_api['verdict']:<10} {_api['score']:<6} {str(_api['latency_sec'])+'s':<10}")
print(f"{'MCP':<6} {_safe(_mcpv,'verdict'):<10} {str(_safe(_mcpv,'score')):<6} {str(_mcp['latency_sec'])+'s':<10}")
print(f"{'A2A':<6} {_safe(_a2av,'verdict'):<10} {str(_safe(_a2av,'score')):<6} {str(_a2a['latency_sec'])+'s':<10}")

## 6. Decision rule + stretch

Pick the **least autonomous protocol** that delivers the outcome with acceptable governance:
1. Deterministic service calls? → **API**
2. AI app needs dynamic tool/context access? → **MCP**
3. Independent agents must coordinate stateful work? → **A2A**
4. Multiple layers? → **Hybrid**: A2A outside, MCP inside agents, APIs underneath.

**Stretch:**
1. Add `@mcp_server.tool()` `get_security_advisories(repo)`. Re-run the agent — host code unchanged.
2. Move to stdio: in a separate `.py` file, `if __name__ == '__main__': mcp_server.run(transport='stdio')`. Connect from Claude Desktop / VS Code with the same tools.
3. Inject a malicious tool result. Add input sanitization at the agent boundary.
4. Replace `CommitsAgent` with a remote A2A endpoint (FastAPI + the A2A protocol).